# Subagents and context isolation

The supervisor receives a `task` tool. A specialist has its own model, tools and messages. The parent gets a result rather than the whole internal dialogue. This example uses `SubAgentMiddleware` directly, so only the explicitly declared specialist is available.

[Subagents](https://docs.langchain.com/oss/python/deepagents/subagents)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from deepagents.backends import StateBackend
from deepagents.middleware.subagents import SubAgentMiddleware
from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.tools import search_handbook
from langchain.agents import create_agent
from model_config import get_model, run_config
from lab.tools import lookup_order


In [ ]:
# Execute
specialist = create_agent(
    model=get_model("subagent"), tools=[search_handbook],
    system_prompt=prompt("subagent_system"),
    middleware=[TraceMiddleware("12_specialist", pretty=True)],
)
agent = create_agent(
    model=get_model(),
    middleware=[
        SubAgentMiddleware(backend=StateBackend(), subagents=[{
            "name": "handbook",
            "description": "Find company policy in the fictional handbook.",
            "runnable": specialist,
        }]),
        TraceMiddleware("12_supervisor", pretty=True),
    ],
)
result = agent.invoke({"messages": [("user", prompt("subagent"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)

Subagents isolate conversation context and tool availability. They do not automatically create a separate process or operating-system security boundary.

### Dictionary-based specialists

A dictionary specialist supplies its own prompt, model, tools and optional middleware. A compiled specialist supplies an existing runnable instead. A parent `task` call sends a task description to the named specialist; only the specialist result returns to the supervisor, keeping intermediate tool messages out of the parent history.

In [ ]:
# Execute
dictionary_agent = create_agent(model=get_model(), middleware=[
    SubAgentMiddleware(backend=StateBackend(), subagents=[{
        "name": "orders", "description": "Look up fictional order delivery status.",
        "system_prompt": "Use lookup_order and report only its delivery facts.",
        "model": get_model("subagent"), "tools": [lookup_order],
    }]), TraceMiddleware("12_dictionary", pretty=True),
])
dictionary_result = dictionary_agent.invoke({"messages": [("user", "Delegate to the orders specialist to look up order DEMO-42.")]}, config=run_config())

In [ ]:
# Inspect output
show_messages(dictionary_result["messages"], pretty=True)
assert any(m.type == "tool" and m.name == "task" for m in dictionary_result["messages"])